# Advanced Agent Patterns: ReAct, Reflection and Memory

Welcome to today's class. Over the next two hours, we will go deep into three powerful patterns that separate basic LLM applications from truly capable AI agents.

Here is what we are covering today:

1. **ReAct Pattern**: How agents interleave reasoning with actions to solve complex tasks
2. **Reflection Pattern**: How agents critique and improve their own outputs
3. **Memory Systems**: How agents remember, retrieve, and use information across interactions

**Why does this matter?**

Think about how products like Google Search, GitHub Copilot, or Amazon Alexa work behind the scenes. They do not just generate text. They reason about what to do, take actions (search the web, run code, query databases), observe the results, and adjust. These are agent patterns in production at massive scale.

By the end of this session, you will understand the mechanics behind these patterns and be able to build them using LangGraph with Gemini for agentic workflows, and the raw Gemini SDK for reflection loops.

Let us start by setting up our environment.

In [ ]:
!pip install -q langgraph langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.3 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"]=userdata.get("GROQ_API_KEY")

In [ ]:
from langchain_groq import ChatGroq
import os
llm=ChatGroq(model="qwen/qwen3-32b",temperature=0,api_key=os.environ["GROQ_API_KEY"])
print(llm.invoke("Say Groq ready").content)

<think>
Okay, the user said "Say Groq ready". I need to respond appropriately. First, I should acknowledge their request. They might be testing if I can respond to specific commands or just want a confirmation. Since the user is asking me to say "Groq ready", I should check if there's any context I'm missing. Maybe they're referring to a specific system or model called Groq? But I'm Qwen, so I need to clarify if they want me to simulate Groq's response or just say that I'm ready. Since the instruction is straightforward, the best approach is to comply and say "Groq ready" as requested. However, I should also add a note explaining that I'm Qwen and not Groq, to avoid confusion. That way, the user knows I'm following their instruction but also clarifying my identity. Let me make sure the response is clear and helpful.
</think>

Groq ready! (Note: I am Qwen, not Groq. Let me know if you'd like assistance with anything!)


In [ ]:
from langchain_groq import ChatGroq
import os
llm=ChatGroq(model="qwen/qwen3-32b",temperature=0,api_key=os.environ["GROQ_API_KEY"])
print(llm.invoke("Groq reflection ready").content)

<think>
Okay, the user mentioned "Groq reflection ready." I need to figure out what they're referring to. Groq is a company known for their high-performance AI chips, right? Maybe they're talking about using Groq's technology for some kind of reflection process. But what does "reflection ready" mean in this context?

Reflection in AI usually refers to the ability of a system to analyze its own processes or decisions. So perhaps the user is asking about integrating Groq's hardware with reflective AI systems. But I'm not sure if Groq has specific products related to reflection. Maybe they're using Groq's chips to enhance the computational power needed for reflection tasks.

Alternatively, "reflection ready" could be a term from a specific framework or project. The user might be referring to a setup where Groq's hardware is prepared to handle reflective computations. I should check if there's any existing documentation or projects that combine Groq with reflection.

Wait, maybe the user i

If you see both "LangGraph ready!" and "Gemini SDK ready!" above, we are all set. We are using two interfaces today for a deliberate reason:

- **LangGraph** is purpose-built for agent loops (ReAct). It manages the cycle of reasoning, tool calling, and observation automatically.
- **Groq via LangChain** gives us full control over individual LLM calls (Reflection). No framework magic, just clean function calls.

Let us dive in.

## ReAct Pattern: Concept Deep Dive

[TIME: 0:10 to 0:30]

# The ReAct Pattern: Reason + Act

ReAct stands for **Reasoning and Acting**. It was introduced in a 2022 paper by Yao et al. and has become one of the foundational patterns for building LLM agents.

### The Core Idea

Most LLMs, when prompted, either:
- **Reason** about a problem (Chain-of-Thought), OR
- **Act** by calling tools or generating outputs

ReAct combines both into a single interleaved loop. The agent cycles through three steps repeatedly:

| Step | What Happens | Example |
|------|-------------|---------|
| **Thought** | The agent reasons about what it knows and what it needs to do next | "The customer is asking about order 1001. Let me look it up." |
| **Action** | The agent calls a tool or takes an external action | `lookup_order("1001")` |
| **Observation** | The agent receives the result of the action | "Order 1001: Shipped, 2 items, arriving June 15" |

This loop repeats until the agent has enough information to produce a final answer.

### Analogy: The Senior Engineer at Amazon

Imagine a senior engineer at Amazon gets paged at 2 AM because checkout is failing for European customers.

Here is how they work through it:

**Thought 1**: "Checkout failures in Europe. Could be a payment gateway issue, a regional CDN problem, or a database timeout. Let me check the payment service logs first."

**Action 1**: Opens CloudWatch, filters payment service logs for EU region.

**Observation 1**: "Payment service is healthy. No errors in the last hour."

**Thought 2**: "Payment is fine. Let me check if the EU load balancer is routing correctly."

**Action 2**: Checks the load balancer dashboard.

**Observation 2**: "The EU-west-1 load balancer shows 90% 5xx errors starting 45 minutes ago."

**Thought 3**: "Found it. The EU load balancer is failing. Let me check what changed recently."

**Action 3**: Checks the deployment history.

**Observation 3**: "A config change was pushed to EU-west-1 at 1:15 AM."

**Final Answer**: "The checkout failure is caused by a bad config deployment to EU-west-1. Rolling back."

This is exactly how a ReAct agent operates. It does not try to answer everything in one shot. It thinks, acts, observes, and refines its understanding iteratively.

### A Concrete ReAct Trace

Let us look at what a ReAct trace looks like for an e-commerce support scenario:

**Question**: "A customer says their order 1005 has not arrived. They want to know what they ordered and whether the items are still available for a replacement."

```
Thought 1: I need to first look up order 1005 to see its status and contents.
Action 1:  lookup_order("1005")
Observation 1: Order 1005 - Status: Delayed, Items: ["Wireless Mouse", "USB-C Hub"],
               Customer: Jane Smith

Thought 2: The order is delayed. The customer wants to know if replacements are
            available. Let me check inventory for "Wireless Mouse" first.
Action 2:  check_inventory("Wireless Mouse")
Observation 2: Wireless Mouse - In Stock: 45 units, Warehouse: Seattle-W2

Thought 3: Wireless Mouse is available. Now let me check "USB-C Hub".
Action 3:  check_inventory("USB-C Hub")
Observation 3: USB-C Hub - In Stock: 3 units, Warehouse: Austin-W1

Thought 4: Both items are in stock, though USB-C Hub is running low.
            I can now provide a complete answer.
Final Answer: Order 1005 (Wireless Mouse and USB-C Hub) is currently delayed.
              Both items are available for replacement. Note that USB-C Hub
              has only 3 units left, so a replacement should be processed quickly.
```

Notice how the agent does not guess at inventory levels. It retrieves real data at each step and reasons about what to do next based on what it found.

### How ReAct Compares to Other Patterns

You have already seen Chain-of-Thought (CoT) prompting. Let us compare it with two other patterns.

| Feature | Chain-of-Thought (CoT) | Plan-and-Solve | ReAct |
|---------|----------------------|----------------|-------|
| **Approach** | Think step by step, then answer | Plan all steps first, then execute sequentially | Interleave thinking and acting |
| **Tool Use** | No | Yes (after planning) | Yes (during reasoning) |
| **Adaptability** | Low. Plan is implicit and fixed | Medium. Plan is explicit but rigid | High. Each step adapts to new observations |
| **Error Recovery** | Poor. Cannot correct mid-reasoning | Moderate. Can re-plan but costly | Strong. Adjusts after every observation |
| **Best For** | Math, logic, simple reasoning | Well-defined multi-step tasks | Open-ended research, dynamic problems |
| **Real-World Analogy** | Solving a math problem on paper | Following a recipe | Debugging a live production system |

**Key insight**: CoT is pure reasoning with no external actions. Plan-and-Solve creates a full plan upfront and then executes it. ReAct is adaptive, meaning it decides what to do next based on what it just learned. This makes ReAct more robust for real-world tasks where the path to the answer is not known in advance.

### QUIZ 1

Look at the following lines from an agent trace and type your answer in chat. Classify each line as **Thought (T)**, **Action (A)**, or **Observation (O)**.

```
Line 1: "The customer wants to know if their laptop is still under warranty.
         Let me look up the order first."
Line 2: lookup_order("2010")
Line 3: "Order 2010: Laptop Pro 15, purchased 8 months ago, Status: Delivered"
Line 4: "The laptop was purchased 8 months ago. Most warranties are 12 months.
         Let me check the product details to confirm the warranty period."
```

Format your answer like: `1-T, 2-A, 3-O, 4-T` (or whatever you think it is).

**Answer to Quiz 1:**

- Line 1: **Thought** (reasoning about what to do)
- Line 2: **Action** (calling a tool)
- Line 3: **Observation** (result from the tool)
- Line 4: **Thought** (reasoning about the observation and planning the next action)

Great. Now let us build this pattern with real code.

## ReAct Pattern: Runnable Demo

[TIME: 0:30 to 0:50]

# Building a ReAct Agent with Custom Tools

We will build a ReAct agent that acts as an e-commerce support assistant. It has access to three tools that query our (simulated) company systems:

1. **lookup_order**: Gets order details by order ID
2. **get_product_details**: Gets product information by product name
3. **check_inventory**: Checks stock levels for a product

All three tools use hardcoded dictionaries, so there are no external API calls and zero chance of network failures. Let us build the data and tools.

In [ ]:
# Our simulated e-commerce database

ORDERS_DB = {
    "1001": {
        "order_id": "1001",
        "customer": "Alice Johnson",
        "items": ["Laptop Pro 15", "Wireless Mouse"],
        "status": "Delivered",
        "order_date": "2024-11-20",
        "delivery_date": "2024-11-25",
        "total": 1349.98
    },
    "1002": {
        "order_id": "1002",
        "customer": "Bob Martinez",
        "items": ["USB-C Hub", "Mechanical Keyboard"],
        "status": "Shipped",
        "order_date": "2024-12-01",
        "estimated_delivery": "2024-12-07",
        "total": 214.98
    },
    "1003": {
        "order_id": "1003",
        "customer": "Carol Lee",
        "items": ["Noise Cancelling Headphones", "Laptop Pro 15", "Phone Stand"],
        "status": "Processing",
        "order_date": "2024-12-05",
        "total": 1479.97
    },
    "1004": {
        "order_id": "1004",
        "customer": "David Kim",
        "items": ["Wireless Mouse", "Phone Stand"],
        "status": "Cancelled",
        "order_date": "2024-11-15",
        "cancellation_reason": "Customer requested cancellation",
        "total": 74.98
    },
    "1005": {
        "order_id": "1005",
        "customer": "Eva Chen",
        "items": ["Mechanical Keyboard", "Monitor 27 inch"],
        "status": "Delayed",
        "order_date": "2024-11-28",
        "delay_reason": "Warehouse backlog",
        "total": 579.98
    }
}

PRODUCTS_DB = {
    "Laptop Pro 15": {
        "name": "Laptop Pro 15",
        "price": 1299.99,
        "category": "Computers",
        "warranty": "24 months",
        "description": "15-inch professional laptop with 16GB RAM and 512GB SSD",
        "rating": 4.7
    },
    "Wireless Mouse": {
        "name": "Wireless Mouse",
        "price": 49.99,
        "category": "Accessories",
        "warranty": "12 months",
        "description": "Ergonomic wireless mouse with 3 DPI settings",
        "rating": 4.3
    },
    "USB-C Hub": {
        "name": "USB-C Hub",
        "price": 89.99,
        "category": "Accessories",
        "warranty": "12 months",
        "description": "7-in-1 USB-C hub with HDMI, USB 3.0, and SD card reader",
        "rating": 4.5
    },
    "Mechanical Keyboard": {
        "name": "Mechanical Keyboard",
        "price": 124.99,
        "category": "Accessories",
        "warranty": "24 months",
        "description": "RGB mechanical keyboard with Cherry MX Blue switches",
        "rating": 4.6
    },
    "Noise Cancelling Headphones": {
        "name": "Noise Cancelling Headphones",
        "price": 129.99,
        "category": "Audio",
        "warranty": "18 months",
        "description": "Over-ear headphones with active noise cancellation and 30hr battery",
        "rating": 4.8
    },
    "Phone Stand": {
        "name": "Phone Stand",
        "price": 24.99,
        "category": "Accessories",
        "warranty": "6 months",
        "description": "Adjustable aluminum phone and tablet stand",
        "rating": 4.2
    },
    "Monitor 27 inch": {
        "name": "Monitor 27 inch",
        "price": 454.99,
        "category": "Displays",
        "warranty": "36 months",
        "description": "27-inch 4K IPS monitor with USB-C input and 99% sRGB",
        "rating": 4.6
    }
}

INVENTORY_DB = {
    "Laptop Pro 15": {"in_stock": 12, "warehouse": "Seattle-W1"},
    "Wireless Mouse": {"in_stock": 145, "warehouse": "Seattle-W2"},
    "USB-C Hub": {"in_stock": 3, "warehouse": "Austin-W1"},
    "Mechanical Keyboard": {"in_stock": 67, "warehouse": "Seattle-W2"},
    "Noise Cancelling Headphones": {"in_stock": 0, "warehouse": "Austin-W1"},
    "Phone Stand": {"in_stock": 230, "warehouse": "Seattle-W1"},
    "Monitor 27 inch": {"in_stock": 8, "warehouse": "Austin-W2"}
}

print(f"Database loaded: {len(ORDERS_DB)} orders, {len(PRODUCTS_DB)} products, {len(INVENTORY_DB)} inventory records")

Database loaded: 5 orders, 7 products, 7 inventory records


Now let us wrap these databases into LangChain-compatible tools. We use the `@tool` decorator, which is the simplest way to create tools in LangChain. The docstring of each function is critical because the LLM reads it to understand when and how to use the tool.

In [ ]:
from langchain_core.tools import tool

@tool
def lookup_order(order_id: str) -> str:
    """Look up an order by its order ID. Returns order details including
    customer name, items ordered, order status, dates, and total amount.
    Use this when you need to find information about a specific order."""

    order = ORDERS_DB.get(order_id)
    if not order:
        return f"Error: No order found with ID '{order_id}'. Valid order IDs are: {list(ORDERS_DB.keys())}"

    result = f"Order {order['order_id']}:\n"
    result += f"  Customer: {order['customer']}\n"
    result += f"  Items: {', '.join(order['items'])}\n"
    result += f"  Status: {order['status']}\n"
    result += f"  Order Date: {order['order_date']}\n"
    result += f"  Total: ${order['total']}\n"

    if 'delivery_date' in order:
        result += f"  Delivery Date: {order['delivery_date']}\n"
    if 'estimated_delivery' in order:
        result += f"  Estimated Delivery: {order['estimated_delivery']}\n"
    if 'delay_reason' in order:
        result += f"  Delay Reason: {order['delay_reason']}\n"
    if 'cancellation_reason' in order:
        result += f"  Cancellation Reason: {order['cancellation_reason']}\n"

    return result

@tool
def get_product_details(product_name: str) -> str:
    """Get detailed information about a product by its name. Returns price,
    category, warranty period, description, and customer rating.
    Use this when you need to know product specifications, pricing, or warranty info."""

    product = PRODUCTS_DB.get(product_name)
    if not product:
        available = list(PRODUCTS_DB.keys())
        return f"Error: Product '{product_name}' not found. Available products: {available}"

    result = f"Product: {product['name']}\n"
    result += f"  Price: ${product['price']}\n"
    result += f"  Category: {product['category']}\n"
    result += f"  Warranty: {product['warranty']}\n"
    result += f"  Description: {product['description']}\n"
    result += f"  Rating: {product['rating']}/5.0\n"

    return result

@tool
def check_inventory(product_name: str) -> str:
    """Check the current inventory level for a product. Returns the number
    of units in stock and the warehouse location.
    Use this when you need to know if a product is available or how many units remain."""

    inventory = INVENTORY_DB.get(product_name)
    if not inventory:
        available = list(INVENTORY_DB.keys())
        return f"Error: Product '{product_name}' not found in inventory. Available products: {available}"

    status = "In Stock" if inventory["in_stock"] > 0 else "Out of Stock"
    low_stock_warning = " (LOW STOCK)" if 0 < inventory["in_stock"] <= 5 else ""

    result = f"Inventory for {product_name}:\n"
    result += f"  Status: {status}{low_stock_warning}\n"
    result += f"  Units Available: {inventory['in_stock']}\n"
    result += f"  Warehouse: {inventory['warehouse']}\n"

    return result

# Quick test to make sure all tools work
print("Testing lookup_order:")
print(lookup_order.invoke({"order_id": "1001"}))
print("\nTesting get_product_details:")
print(get_product_details.invoke({"product_name": "Wireless Mouse"}))
print("\nTesting check_inventory:")
print(check_inventory.invoke({"product_name": "Noise Cancelling Headphones"}))

Testing lookup_order:
Order 1001:
  Customer: Alice Johnson
  Items: Laptop Pro 15, Wireless Mouse
  Status: Delivered
  Order Date: 2024-11-20
  Total: $1349.98
  Delivery Date: 2024-11-25


Testing get_product_details:
Product: Wireless Mouse
  Price: $49.99
  Category: Accessories
  Warranty: 12 months
  Description: Ergonomic wireless mouse with 3 DPI settings
  Rating: 4.3/5.0


Testing check_inventory:
Inventory for Noise Cancelling Headphones:
  Status: Out of Stock
  Units Available: 0
  Warehouse: Austin-W1



All three tools are working. Notice a few design choices:

- Each tool returns a **formatted string**, not a dictionary. This makes it easier for the LLM to read and reason about the results.
- Each tool has **error handling** that tells the LLM what valid inputs look like. This helps the agent self-correct if it guesses a wrong product name or order ID.
- The **docstrings are detailed**. The LLM uses these to decide which tool to call and when.

Now let us create the ReAct agent.

In [ ]:
from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent
import os
llm=ChatGroq(model="qwen/qwen3-32b",temperature=0,api_key=os.environ["GROQ_API_KEY"])
tools=[lookup_order,get_product_details,check_inventory]
agent=create_react_agent(model=llm,tools=tools,prompt="You are a helpful e-commerce customer support agent.")
print("ReAct agent created.")

ReAct agent created.


/tmp/ipykernel_16995/1586022328.py:6: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent=create_react_agent(model=llm,tools=tools,prompt="You are a helpful e-commerce customer support agent.")


Let us create a helper function that runs the agent and prints each step clearly so we can see the full ReAct trace.

In [ ]:
def run_agent(query: str):
    """Run the ReAct agent and display the full trace of steps."""

    print("=" * 70)
    print(f"QUERY: {query}")
    print("=" * 70)

    step_count = 0
    tool_call_count = 0

    for step in agent.stream(
        {"messages": [{"role": "user", "content": query}]},
        stream_mode="updates"
    ):
        for node_name, node_output in step.items():
            messages = node_output.get("messages", [])
            for msg in messages:
                msg_type = msg.__class__.__name__

                if msg_type == "AIMessage":
                    # The agent's reasoning or final answer
                    if msg.content:
                        step_count += 1
                        print(f"\n[Step {step_count} - Agent Reasoning]")
                        print(msg.content)

                    # Check for tool calls
                    if hasattr(msg, "tool_calls") and msg.tool_calls:
                        for tc in msg.tool_calls:
                            tool_call_count += 1
                            print(f"\n[Step {step_count} - Tool Call #{tool_call_count}]")
                            print(f"  Tool: {tc['name']}")
                            print(f"  Input: {tc['args']}")

                elif msg_type == "ToolMessage":
                    print(f"\n[Observation from {msg.name}]")
                    print(f"  {msg.content}")

    print(f"\n{'=' * 70}")
    print(f"Trace complete. Tool calls made: {tool_call_count}")
    print("=" * 70)

### Query 1: Simple Single-Tool Lookup

Let us start with a straightforward question that requires just one tool call.

In [ ]:
run_agent("What is the status of order 1002? Who placed it and when will it arrive?")

QUERY: What is the status of order 1002? Who placed it and when will it arrive?

[Step 0 - Tool Call #1]
  Tool: lookup_order
  Input: {'order_id': '1002'}

[Observation from lookup_order]
  Order 1002:
  Customer: Bob Martinez
  Items: USB-C Hub, Mechanical Keyboard
  Status: Shipped
  Order Date: 2024-12-01
  Total: $214.98
  Estimated Delivery: 2024-12-07


[Step 1 - Agent Reasoning]
The status of order 1002 is **Shipped**. It was placed by **Bob Martinez** on December 1, 2024. The estimated delivery date is **December 7, 2024**. Let me know if you need further details!

Trace complete. Tool calls made: 1


Notice the agent's behavior:
1. It recognized this is an order lookup question
2. It called `lookup_order` with the correct order ID
3. It read the observation and formulated a friendly answer

This is a single-hop ReAct trace. One thought, one action, one observation, and a final answer. Now let us increase the complexity.

### Query 2: Multi-Tool Reasoning

This question requires the agent to use multiple tools and connect information across them.

In [ ]:
run_agent(
    "I placed order 1002. Can you tell me the price and warranty "
    "information for each item in my order?"
)

QUERY: I placed order 1002. Can you tell me the price and warranty information for each item in my order?

[Step 0 - Tool Call #1]
  Tool: lookup_order
  Input: {'order_id': '1002'}

[Observation from lookup_order]
  Order 1002:
  Customer: Bob Martinez
  Items: USB-C Hub, Mechanical Keyboard
  Status: Shipped
  Order Date: 2024-12-01
  Total: $214.98
  Estimated Delivery: 2024-12-07


[Step 0 - Tool Call #2]
  Tool: get_product_details
  Input: {'product_name': 'USB-C Hub'}

[Step 0 - Tool Call #3]
  Tool: get_product_details
  Input: {'product_name': 'Mechanical Keyboard'}

[Observation from get_product_details]
  Product: USB-C Hub
  Price: $89.99
  Category: Accessories
  Warranty: 12 months
  Description: 7-in-1 USB-C hub with HDMI, USB 3.0, and SD card reader
  Rating: 4.5/5.0


[Observation from get_product_details]
  Product: Mechanical Keyboard
  Price: $124.99
  Category: Accessories
  Warranty: 24 months
  Description: RGB mechanical keyboard with Cherry MX Blue switches
  R

This is where ReAct gets interesting. The agent had to:

1. First look up order 1002 to find out what items were ordered
2. Then call `get_product_details` for each item separately
3. Finally synthesize all the information into a coherent response

The agent decided what to do at each step based on what it learned in the previous step. It could not have planned all the tool calls upfront because it did not know the items until after the first lookup.

### Query 3: Complex Multi-Step Reasoning

Now let us give the agent a question that requires reasoning across all three tools.

In [ ]:
run_agent(
    "I have order 1003. Can you check which of my items are currently "
    "in stock and which ones might be hard to replace if something goes wrong? "
    "Also, what is the total warranty coverage I have?"
)

QUERY: I have order 1003. Can you check which of my items are currently in stock and which ones might be hard to replace if something goes wrong? Also, what is the total warranty coverage I have?

[Step 0 - Tool Call #1]
  Tool: lookup_order
  Input: {'order_id': '1003'}

[Observation from lookup_order]
  Order 1003:
  Customer: Carol Lee
  Items: Noise Cancelling Headphones, Laptop Pro 15, Phone Stand
  Status: Processing
  Order Date: 2024-12-05
  Total: $1479.97


[Step 0 - Tool Call #2]
  Tool: check_inventory
  Input: {'product_name': 'Noise Cancelling Headphones'}

[Step 0 - Tool Call #3]
  Tool: check_inventory
  Input: {'product_name': 'Laptop Pro 15'}

[Step 0 - Tool Call #4]
  Tool: check_inventory
  Input: {'product_name': 'Phone Stand'}

[Observation from check_inventory]
  Inventory for Noise Cancelling Headphones:
  Status: Out of Stock
  Units Available: 0
  Warehouse: Austin-W1


[Observation from check_inventory]
  Inventory for Laptop Pro 15:
  Status: In Stock
  Unit

This query pushed the agent through a multi-step process:

1. Look up order 1003 to get the list of items
2. Check inventory for each item (3 separate calls)
3. Get product details for warranty information (3 separate calls)
4. Reason across all the results to identify which items are at risk and calculate warranty coverage

This is the power of ReAct. The agent navigated through potentially 7 tool calls, adapted at each step, and produced a synthesized answer that no single tool call could have provided.

### Query 4: Handling Edge Cases

Let us see how the agent handles a tricky situation.

In [ ]:
run_agent(
    "I want to reorder everything from order 1004. Is that possible? "
    "Check if all items are available."
)

QUERY: I want to reorder everything from order 1004. Is that possible? Check if all items are available.

[Step 0 - Tool Call #1]
  Tool: lookup_order
  Input: {'order_id': '1004'}

[Observation from lookup_order]
  Order 1004:
  Customer: David Kim
  Items: Wireless Mouse, Phone Stand
  Status: Cancelled
  Order Date: 2024-11-15
  Total: $74.98
  Cancellation Reason: Customer requested cancellation


[Step 0 - Tool Call #2]
  Tool: check_inventory
  Input: {'product_name': 'Wireless Mouse'}

[Step 0 - Tool Call #3]
  Tool: check_inventory
  Input: {'product_name': 'Phone Stand'}

[Observation from check_inventory]
  Inventory for Phone Stand:
  Status: In Stock
  Units Available: 230
  Warehouse: Seattle-W1


[Observation from check_inventory]
  Inventory for Wireless Mouse:
  Status: In Stock
  Units Available: 145
  Warehouse: Seattle-W2


[Step 1 - Agent Reasoning]
Both items from your cancelled order (Order 1004) are currently in stock:
- **Wireless Mouse**: 145 units available in

This is interesting because order 1004 was cancelled. The agent had to:
1. Look up the cancelled order
2. Recognize it was cancelled and understand the customer wants to re-place it
3. Check inventory for each item
4. Give a recommendation based on availability

The agent handled the edge case naturally because ReAct lets it reason about unexpected situations at each step.

### QUIZ 2

Here is a new query I am about to run:

**"Compare the ratings and prices of the Mechanical Keyboard and the Noise Cancelling Headphones. Which one is the better value for money?"**

Before I run this, type in chat: **How many tool calls do you think the agent will make?** Pick a number between 1 and 4.

In [ ]:
run_agent(
    "Compare the ratings and prices of the Mechanical Keyboard and the "
    "Noise Cancelling Headphones. Which one is the better value for money?"
)

QUERY: Compare the ratings and prices of the Mechanical Keyboard and the Noise Cancelling Headphones. Which one is the better value for money?

[Step 0 - Tool Call #1]
  Tool: get_product_details
  Input: {'product_name': 'Mechanical Keyboard'}

[Step 0 - Tool Call #2]
  Tool: get_product_details
  Input: {'product_name': 'Noise Cancelling Headphones'}

[Observation from get_product_details]
  Product: Mechanical Keyboard
  Price: $124.99
  Category: Accessories
  Warranty: 24 months
  Description: RGB mechanical keyboard with Cherry MX Blue switches
  Rating: 4.6/5.0


[Observation from get_product_details]
  Product: Noise Cancelling Headphones
  Price: $129.99
  Category: Audio
  Warranty: 18 months
  Description: Over-ear headphones with active noise cancellation and 30hr battery
  Rating: 4.8/5.0


[Step 1 - Agent Reasoning]
The **Noise Cancelling Headphones** ($129.99, 4.8/5.0) and **Mechanical Keyboard** ($124.99, 4.6/5.0) are both highly rated, but here's the comparison:

- **P

The agent needed **2 tool calls**, one `get_product_details` for each product. It then compared them using pure reasoning (no tool needed for the comparison itself). If you predicted 2, well done.

Notice that the agent did not need to check inventory or look up orders. It correctly identified that only product details were relevant to a "value for money" comparison. This is the adaptive nature of ReAct. The agent uses only the tools it needs based on the specific question.

Now let us move on to the second major pattern: Reflection.

## Reflection: Concept Deep Dive

[TIME: 0:50 to 1:05]

# Self-Reflection and Critique Mechanisms

So far, with ReAct, we have seen how agents can gather external information. But what about improving the quality of their own outputs? This is where **reflection** comes in.

### The Core Idea

Reflection is a pattern where an agent evaluates its own output, identifies weaknesses, and generates an improved version. Think of it as a built-in quality assurance loop.

The simplest form is the **Judge-Revise** pattern:

```
Generator  --->  Initial Output
                      |
                      v
Judge      --->  Critique ("Here is what is wrong...")
                      |
                      v
Generator  --->  Revised Output (addressing the critique)
                      |
                      v
               (optionally repeat)
```

### Analogy: Netflix's Culture of Retrospection

Netflix is famous for its engineering culture of blameless post-mortems. After every significant incident:

1. **The team produces** an incident report (this is the "generation" step).
2. **A review group critiques** the report: Was the root cause correctly identified? Were the mitigation steps sufficient? Are there gaps in monitoring? (This is the "judge" step.)
3. **The team revises** the report and the actual systems based on the feedback (this is the "revise" step).

This cycle is what makes Netflix's systems increasingly reliable over time. The same principle applies to LLM agents. Without reflection, an agent produces its first-draft output and stops. With reflection, the agent can catch its own mistakes, fill in gaps, and produce higher-quality results.

### The Judge-Revise Pattern in Detail

There are two common architectures for reflection:

**Architecture 1: Single LLM, Two Roles**

The same LLM plays both the generator and the judge, but with different system prompts.

```
[System: You are a code generator]  --->  Generates code
[System: You are a code reviewer]   --->  Critiques the code
[System: You are a code generator]  --->  Revises based on critique
```

This is simpler but the model might be blind to its own systematic errors.

**Architecture 2: Two Different LLMs**

A separate, possibly stronger model acts as the judge.

```
Model A (Generator)  --->  Generates code
Model B (Judge)      --->  Critiques the code
Model A (Generator)  --->  Revises based on Model B's critique
```

This is more robust because the judge has a different "perspective." In practice at companies like Google or Microsoft, you often see a stronger model (like GPT-4 or Gemini Pro) judging outputs from a faster, cheaper model.

Today we will implement Architecture 1 using the raw Gemini SDK, with the same model playing both roles through different system instructions.

### When Reflection Helps vs. When It Does Not

Reflection is not a magic bullet. Here is when it works well and when it does not.

| Reflection Helps | Reflection Does Not Help |
|-----------------|------------------------|
| Code has logical bugs the LLM can spot on review | The task requires knowledge the LLM does not have |
| Output misses requirements stated in the prompt | The model consistently makes the same error (systematic bias) |
| Writing quality can be improved with a second pass | The output is already near-optimal |
| Formatting or structural issues | Latency is a hard constraint (reflection adds rounds) |

**Key insight**: Reflection is most valuable when the generator's errors are the kind that become obvious upon review. This is similar to how you often spot bugs in your own code when you come back to review it after a break.

### QUIZ 3

Which of the following scenarios would benefit MOST from a reflection pattern? Type A, B, or C in chat.

**A.** An agent that needs to translate English to French. The LLM consistently mistranslates idiomatic expressions.

**B.** An agent that generates SQL queries. Sometimes it forgets a WHERE clause or uses the wrong JOIN type.

**C.** An agent that needs to predict stock prices based on historical data.

**Answer: B**

- **A** describes a systematic error. The model keeps making the same idiomatic mistakes, meaning reflection is unlikely to fix it because the model does not "know" the correct translation.
- **B** is a classic case where reflection shines. Forgetting a WHERE clause or using the wrong JOIN type are errors that become obvious when you re-read the query. A judge prompt like "Check if this query correctly filters the data and uses appropriate joins" would catch these.
- **C** is not a generation quality problem. It is a fundamental capability limitation. No amount of self-reflection will make an LLM good at stock prediction.

Now let us build it.

## Reflection: Runnable Demo

[TIME: 1:05 to 1:25]

# Building a Judge-Revise Loop for Code Generation

We will build a system where:
1. A **Generator** writes a Python function based on a specification
2. A **Judge** reviews the code for correctness, edge cases, and style
3. The **Generator** revises the code based on the judge's feedback
4. We can run this loop multiple times to see iterative improvement

We are using the raw Gemini SDK here. No LangChain, no chains, no abstractions. Just direct function calls to the model with different system instructions. This makes the reflection pattern completely transparent.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage,HumanMessage
import os
llm=ChatGroq(model="qwen/qwen3-32b",temperature=0,api_key=os.environ["GROQ_API_KEY"])
def _ask(sys,u): return llm.invoke([SystemMessage(content=sys),HumanMessage(content=u)]).content

def generate_code(spec): return _ask("You are an expert Python developer. Return only Python code.",spec)
def judge_code(code): return _ask("You are a senior reviewer. Reply LGTM if perfect.",code)
def revise_code(spec,previous_code,feedback): return _ask("You are an expert Python developer.",f"Specification:\n{spec}\n\nPrevious Code:\n{previous_code}\n\nFeedback:\n{feedback}")

All three functions are working. Notice how simple each one is. There is no framework magic here. Each function:

1. Creates a model instance with a specific `system_instruction`
2. Sends a prompt with `generate_content`
3. Returns the text response

This is the raw building block of the Judge-Revise pattern. Now let us build the loop.

In [ ]:
def reflect_and_revise(spec: str, max_iterations: int = 3, verbose: bool = True) -> dict:
    """
    Run the Judge-Revise reflection loop.

    Args:
        spec: The code specification/requirements
        max_iterations: Maximum number of revision cycles
        verbose: Whether to print each step

    Returns:
        Dictionary with the history of generations, critiques, and the final code
    """
    history = {"iterations": []}

    # Step 1: Initial Generation
    if verbose:
        print("=" * 70)
        print("INITIAL GENERATION")
        print("=" * 70)

    current_code = generate_code(spec)

    if verbose:
        print(current_code)

    for i in range(max_iterations):
        if verbose:
            print(f"\n{'=' * 70}")
            print(f"JUDGE REVIEW - ITERATION {i + 1}")
            print(f"{'=' * 70}")

        # Step 2: Judge reviews the code
        critique = judge_code(current_code)

        if verbose:
            print(f"\nFEEDBACK:\n{critique}")

        iteration_record = {
            "iteration": i + 1,
            "code": current_code,
            "critique": critique
        }
        history["iterations"].append(iteration_record)

        # Check if the judge approves
        if "LGTM" in critique.strip().upper():
            if verbose:
                print(f"\nJudge approved the code at iteration {i + 1}!")
            history["final_code"] = current_code
            history["total_iterations"] = i + 1
            history["approved"] = True
            return history

        # Step 3: Revise based on feedback
        if verbose:
            print(f"\n{'=' * 70}")
            print(f"REVISION - ITERATION {i + 1}")
            print(f"{'=' * 70}")

        revised_code = revise_code(spec, current_code, critique)

        if verbose:
            print(f"\nREVISED CODE:\n{revised_code}")

        current_code = revised_code

    history["final_code"] = current_code
    history["total_iterations"] = max_iterations
    history["approved"] = False
    return history

print("Reflection loop is ready.")

Reflection loop is ready.


Let us test this with a specification that is complex enough to trigger meaningful critique on the first attempt.

In [ ]:
spec_1 = """
Write a Python function called 'merge_sorted_lists' that takes two sorted lists
of integers and returns a single sorted list containing all elements from both lists.

Requirements:
- Handle empty lists
- Handle lists of different lengths
- Handle duplicate values
- Do NOT use Python's built-in sort() or sorted()
- Time complexity should be O(n + m) where n and m are the lengths of the input lists
"""

result_1 = reflect_and_revise(spec_1, max_iterations=3)

INITIAL GENERATION
<think>
Okay, I need to write a Python function called merge_sorted_lists that takes two sorted lists and returns a merged sorted list. Let me think about how to approach this.

The problem says not to use the built-in sort or sorted functions. Oh right, this is probably meant to implement the merge step from the merge sort algorithm. Because merge sort's merge step does exactly that—combines two sorted lists into one in O(n + m) time.

So the plan is to use two pointers, one for each list. Compare the current elements of both lists, take the smaller one, add it to the result, and move the pointer. Repeat until all elements are processed.

Let me outline the steps:

Initialize an empty list to hold the merged result. Let's call it merged_list.

Initialize two pointers, i and j, starting at 0 for list1 and list2 respectively.

Loop while i is less than len(list1) and j is less than len(list2). Compare list1[i] and list2[j]. If list1[i] is smaller, append it to merged_

Let us examine how the code improved across iterations.

In [ ]:
print(f"Total review iterations: {result_1['total_iterations']}")
print(f"Judge approved: {result_1['approved']}")

for iteration in result_1["iterations"]:
    print(f"\n{'=' * 50}")
    print(f"Iteration {iteration['iteration']} - Critique Summary:")
    print(f"{'=' * 50}")
    # Show first 400 characters of the critique
    critique_preview = iteration["critique"][:400]
    print(critique_preview)
    if len(iteration["critique"]) > 400:
        print("...")

print(f"\n{'=' * 50}")
print("FINAL VERSION:")
print(f"{'=' * 50}")
print(result_1["final_code"])

Total review iterations: 1
Judge approved: True

Iteration 1 - Critique Summary:
<think>
Okay, I need to review the provided Python function `merge_sorted_lists`. Let me start by understanding what it's supposed to do. The function takes two sorted lists and merges them into a single sorted list without using built-in sort functions. 

Looking at the code, it uses two pointers `i` and `j` to iterate through `list1` and `list2`, respectively. The main loop compares elements at 
...

FINAL VERSION:
<think>
Okay, I need to write a Python function called merge_sorted_lists that takes two sorted lists and returns a merged sorted list. Let me think about how to approach this.

The problem says not to use the built-in sort or sorted functions. Oh right, this is probably meant to implement the merge step from the merge sort algorithm. Because merge sort's merge step does exactly that—combines two sorted lists into one in O(n + m) time.

So the plan is to use two pointers, one for each list. Co

### Let us Verify the Final Code Actually Works

One of the strengths of code generation with reflection is that we can actually test the output. Let us execute the final code and run some test cases.

In [ ]:
import re

# Extract only the Python code from the final_code string
code_match = re.search(r"```python\n(.*?)```", result_1["final_code"], re.DOTALL)
if code_match:
    executable_code = code_match.group(1)
else:
    executable_code = result_1["final_code"] # Fallback if regex fails, though it might still error

# Execute the final generated code
exec(executable_code)

# Run test cases
print("Test 1 - Normal case:")
print(f"  merge_sorted_lists([1, 3, 5], [2, 4, 6]) = {merge_sorted_lists([1, 3, 5], [2, 4, 6])}")

print("\nTest 2 - Empty lists:")
print(f"  merge_sorted_lists([], [1, 2, 3]) = {merge_sorted_lists([], [1, 2, 3])}")
print(f"  merge_sorted_lists([1, 2], []) = {merge_sorted_lists([1, 2], [])}")
print(f"  merge_sorted_lists([], []) = {merge_sorted_lists([], [])}")

print("\nTest 3 - Duplicates:")
print(f"  merge_sorted_lists([1, 2, 2], [2, 3, 3]) = {merge_sorted_lists([1, 2, 2], [2, 3, 3])}")

print("\nTest 4 - Different lengths:")
print(f"  merge_sorted_lists([1], [2, 3, 4, 5]) = {merge_sorted_lists([1], [2, 3, 4, 5])}")

Test 1 - Normal case:
  merge_sorted_lists([1, 3, 5], [2, 4, 6]) = [1, 2, 3, 4, 5, 6]

Test 2 - Empty lists:
  merge_sorted_lists([], [1, 2, 3]) = [1, 2, 3]
  merge_sorted_lists([1, 2], []) = [1, 2]
  merge_sorted_lists([], []) = []

Test 3 - Duplicates:
  merge_sorted_lists([1, 2, 2], [2, 3, 3]) = [1, 2, 2, 2, 3, 3]

Test 4 - Different lengths:
  merge_sorted_lists([1], [2, 3, 4, 5]) = [1, 2, 3, 4, 5]


The code works correctly across all test cases. This is a tangible demonstration of why reflection matters. The initial generation might have missed edge cases, but the judge caught them and the revision addressed them.

### QUIZ 4

Let us run another reflection cycle with a different specification. Before I run it, read the spec below and predict in chat: **What is the most likely issue the Judge will flag on the first attempt?**

Type your prediction in chat.

In [ ]:
spec_2 = """
Write a Python function called 'flatten_dict' that takes a nested dictionary
and returns a flattened dictionary where nested keys are joined with a dot separator.

Example:
Input:  {"a": 1, "b": {"c": 2, "d": {"e": 3}}}
Output: {"a": 1, "b.c": 2, "b.d.e": 3}

Requirements:
- Handle arbitrary nesting depth
- Handle empty nested dictionaries
- Handle cases where values are lists (keep lists as-is, do not flatten them)
"""

result_2 = reflect_and_revise(spec_2, max_iterations=3)

INITIAL GENERATION
<think>
Okay, I need to write a Python function called flatten_dict that takes a nested dictionary and returns a flattened version. The keys in the output should be joined with dots. Let me think about how to approach this.

First, the example given is {"a": 1, "b": {"c": 2, "d": {"e": 3}}} which becomes {"a": 1, "b.c": 2, "b.d.e": 3}. So for each nested level, we prepend the parent key with a dot.

The function needs to handle arbitrary nesting. So recursion might be a good approach here. For each key in the input dictionary, if the value is another dictionary, we recursively flatten it. But how to accumulate the keys?

Let me think of a helper function that takes the current dictionary, a parent key, and accumulates into the result. For example, for each key-value pair in the current dict, if the value is a dict, we call the helper again with the parent key being the current key appended to the existing parent. Otherwise, we add the combined key to the result.

Wai

In [ ]:
import re

# Extract only the Python code from the final_code string
code_match = re.search(r"```python\n(.*?)```", result_2["final_code"], re.DOTALL)
if code_match:
    executable_code = code_match.group(1)
else:
    executable_code = result_2["final_code"] # Fallback if regex fails, though it might still error

# Verify the final flatten_dict works
exec(executable_code)

print("Test 1 - Basic nesting:")
print(f"  {flatten_dict({'a': 1, 'b': {'c': 2, 'd': {'e': 3}}})}")

print("\nTest 2 - Empty nested dict:")
print(f"  {flatten_dict({'a': 1, 'b': {}})}")

print("\nTest 3 - Lists preserved:")
print(f"  {flatten_dict({'a': [1, 2, 3], 'b': {'c': [4, 5]}})}")

print("\nTest 4 - Flat dict (no nesting):")
print(f"  {flatten_dict({'x': 10, 'y': 20})}")

print("\nTest 5 - Empty dict:")
print(f"  {flatten_dict({})})")

Test 1 - Basic nesting:
  {'a': 1, 'b.c': 2, 'b.d.e': 3}

Test 2 - Empty nested dict:
  {'a': 1}

Test 3 - Lists preserved:
  {'a': [1, 2, 3], 'b.c': [4, 5]}

Test 4 - Flat dict (no nesting):
  {'x': 10, 'y': 20}

Test 5 - Empty dict:
  {})


Common issues that the Judge typically flags on dictionary flattening functions include: not handling empty nested dictionaries (should they create a key or be skipped?), not handling the case where the input itself is empty, or missing type hints and docstrings. How close was your prediction?

### Key Observations on Reflection

From the two demos above, notice these patterns:

1. **First drafts are often functionally correct but miss edge cases.** The judge catches what the generator overlooks.
2. **Each revision is targeted.** The generator does not rewrite from scratch. It surgically addresses the critique.
3. **Diminishing returns.** The biggest improvements happen in the first revision. By the second or third round, changes become minor (style tweaks, docstring improvements).
4. **The "LGTM" check is important.** Without it, the loop would always run for the maximum iterations, even when the code is already good.

This is the same pattern that makes code review at companies like Google effective. The first review catches the big issues. Subsequent reviews catch progressively smaller ones.

## Memory Systems: Conceptual Deep Dive

[TIME: 1:25 to 1:50]

# Memory Systems for Agents

Here is a fundamental problem with LLM agents: by default, they are stateless. Every time you send a new message, the model has no recollection of previous interactions (unless you explicitly include them in the prompt). This is what we call the **goldfish problem**, meaning the agent forgets everything after each turn.

To build agents that can work on long tasks, learn from past interactions, and maintain context, we need memory systems.

There are three types of memory that matter for agents:

| Memory Type | What It Stores | Duration | Analogy |
|------------|---------------|----------|---------|
| **Short-term Memory** | Current conversation context | Single session | Your working memory while solving a problem |
| **Episodic Memory** | Conversation history across sessions | Medium-term | Your memory of past conversations with a colleague |
| **Long-term Memory** | Accumulated knowledge in a vector store | Persistent | A company's knowledge base or documentation |

Let us examine each one.

### Short-Term Memory: The Context Window

Short-term memory is the simplest form. It is the context window of the LLM itself. Everything in the current prompt (system message, conversation history, tool results) is the agent's short-term memory.

**Analogy: Tesla's Real-Time Sensor Buffer**

Tesla's Autopilot processes data from cameras, radar, and ultrasonic sensors in real-time. At any moment, the system has a buffer of the most recent sensor readings. It does not store every frame from every drive. It works with what is in the current buffer.

Similarly, an LLM agent's short-term memory is limited to what fits in the context window (e.g., 1 million tokens for Gemini 2.0, 128K for GPT-4). Once the conversation exceeds this limit, the oldest information gets truncated or must be summarized.

**The trade-off**: Larger context windows mean more short-term memory, but also higher latency and cost. This is why companies like Google and Anthropic are racing to expand context windows. A 1 million token context window is essentially giving the agent a much larger "working memory."

```
Context Window = [System Prompt] + [Conversation History] + [Current Query]

Available space for memory = Total Window - System Prompt - Current Query
```

### Episodic Memory: Conversation History

Episodic memory stores the record of past interactions. Unlike short-term memory, it persists between sessions. When you come back to ChatGPT the next day and it remembers your previous conversation, that is episodic memory.

**Analogy: Salesforce CRM Storing Customer Interactions**

Salesforce CRM keeps a timeline of every interaction with a customer: emails, calls, meetings, support tickets. When a sales rep picks up a conversation, they can scroll back through the history to understand context.

For agents, episodic memory works the same way:

```
Session 1: User asked about setting up a React project
Session 2: User asked about React state management (agent recalls Session 1 context)
Session 3: User asks about deploying their React app (agent recalls Sessions 1 and 2)
```

Implementation approaches:
- **Full history**: Store every message. Simple but does not scale.
- **Summarized history**: Periodically summarize older conversations. Saves tokens.
- **Windowed history**: Keep only the last N messages. Lossy but efficient.

### Long-Term Memory: Vector Stores

Long-term memory is the most powerful and complex type. It uses a **vector store** (like ChromaDB, Pinecone, or FAISS) to store information as embeddings, which are numerical representations of text that capture semantic meaning.

**Analogy: Spotify's Recommendation Index**

Spotify does not just store songs as audio files. It creates numerical representations (embeddings) of each song that capture features like tempo, mood, genre, and instrumentation. When you ask for "songs like this one," Spotify does not compare audio files. It finds songs whose embeddings are close to the current song's embedding in vector space.

Long-term memory for agents works identically:

1. **Store**: Convert text (documents, past conversations, knowledge) into embeddings
2. **Retrieve**: When the agent needs information, convert the query into an embedding and find the most similar stored embeddings
3. **Use**: Inject the retrieved information into the prompt as context

### Memory Retrieval and Relevance Scoring

The key mechanism that makes long-term memory work is **similarity search**. When the agent needs to recall something, it computes how similar the current query is to each stored memory.

The most common similarity metric is **cosine similarity**:

$$\text{cosine\_similarity}(\vec{A}, \vec{B}) = \frac{\vec{A} \cdot \vec{B}}{||\vec{A}|| \times ||\vec{B}||}$$

Where $\vec{A}$ and $\vec{B}$ are embedding vectors. The result ranges from $-1$ (opposite) to $1$ (identical).

Here is pseudocode showing how a memory-augmented agent retrieves relevant information:

In [ ]:
# PSEUDOCODE: Memory Retrieval System
# This is illustrative and not meant to be executed

class AgentMemory:
    """
    Manages long-term memory for an agent using a vector store.
    """
    def __init__(self, embedding_model, vector_store):
        self.embedding_model = embedding_model  # e.g., Google text-embedding-004
        self.vector_store = vector_store        # e.g., ChromaDB, FAISS, Pinecone

    def store_memory(self, text: str, metadata: dict = None):
        """Store a piece of information in long-term memory."""
        # Convert text to embedding vector (e.g., 768-dimensional float array)
        embedding = self.embedding_model.embed(text)

        # Store in vector database with metadata for filtering
        self.vector_store.add(
            embedding=embedding,
            text=text,
            metadata=metadata  # e.g., {"timestamp": "2024-01-15", "source": "user"}
        )

    def retrieve_relevant_memories(self, query: str, top_k: int = 3) -> list:
        """Retrieve the most relevant memories for a given query."""
        # Convert query to embedding
        query_embedding = self.embedding_model.embed(query)

        # Search vector store using cosine similarity
        results = self.vector_store.similarity_search(
            query_embedding=query_embedding,
            top_k=top_k
        )

        # Results arrive sorted by similarity score (highest first)
        # Example results:
        # [("User prefers Python over Java", 0.92),
        #  ("User is working on a web scraping project", 0.87),
        #  ("User's favorite food is pizza", 0.12)]

        # Filter by relevance threshold to avoid injecting noise
        relevant = [text for text, score in results if score > 0.5]
        return relevant

print("Pseudocode: Memory retrieval system architecture")

Pseudocode: Memory retrieval system architecture


### How These Memory Types Work Together

In a production agent, all three memory types work in concert. Here is a pseudocode example showing a complete memory-augmented agent:

In [ ]:
# PSEUDOCODE: Complete Memory-Augmented Agent
# This is illustrative and not meant to be executed

class MemoryAugmentedAgent:
    """
    An agent that uses all three memory types:
    short-term (context window), episodic (conversation history),
    and long-term (vector store).
    """
    def __init__(self, llm, memory: "AgentMemory"):
        self.llm = llm
        self.memory = memory
        self.conversation_history = []  # Episodic memory (in-session)

    def respond(self, user_message: str) -> str:
        # 1. LONG-TERM MEMORY: Retrieve relevant past knowledge
        relevant_memories = self.memory.retrieve_relevant_memories(
            query=user_message,
            top_k=3
        )

        # 2. EPISODIC MEMORY: Include recent conversation turns
        recent_history = self.conversation_history[-10:]

        # 3. SHORT-TERM MEMORY: Everything assembled into the prompt
        #    below constitutes the agent's working memory
        prompt = (
            "You are a helpful assistant with access to the following context:\n\n"
            "RELEVANT KNOWLEDGE (from past interactions):\n"
            f"{relevant_memories}\n\n"
            "RECENT CONVERSATION:\n"
            f"{recent_history}\n\n"
            "USER MESSAGE:\n"
            f"{user_message}\n\n"
            "Respond helpfully, using the provided context when relevant."
        )

        # Generate response
        response = self.llm.generate(prompt)

        # Update episodic memory for this session
        self.conversation_history.append(f"User: {user_message}")
        self.conversation_history.append(f"Agent: {response}")

        # Persist to long-term memory for future sessions
        self.memory.store_memory(
            text=f"User asked: {user_message}. Agent responded: {response}",
            metadata={"type": "conversation", "timestamp": "2024-07-01"}
        )

        return response

print("Pseudocode: Memory-augmented agent architecture")

Pseudocode: Memory-augmented agent architecture


### Real-World Memory Architectures

Different companies use these memory types in different proportions depending on their use case:

| Product | Short-term | Episodic | Long-term | Why |
|---------|-----------|----------|-----------|-----|
| **ChatGPT** | Large context window | Conversation history, "Memory" feature | Not directly (but fine-tuning serves a similar purpose) | General assistant needs to remember user preferences |
| **GitHub Copilot** | Current file + open tabs | Not used heavily | Codebase embeddings in Copilot Workspace | Code completion needs local context, not conversation history |
| **Google Search (AI Overviews)** | Current query | Previous searches in session | Entire web index (massive vector store) | Search needs broad knowledge retrieval |
| **Customer Support Bots (e.g., Intercom)** | Current ticket | Full customer interaction timeline | Company knowledge base, FAQ embeddings | Support needs both customer history and product knowledge |

### QUIZ 5

Match each scenario to the PRIMARY memory type it relies on. Type your answers in chat as three pairs (e.g., "1-A, 2-B, 3-C").

**Scenarios:**
1. A coding assistant that remembers you prefer TypeScript over JavaScript across sessions
2. A chatbot that keeps track of what you have said earlier in the current conversation
3. A research agent that searches through 10,000 stored research papers to find relevant ones

**Memory Types:**
- A. Short-term Memory (context window)
- B. Episodic Memory (conversation history)
- C. Long-term Memory (vector store)

**Answers:**
1. **B (Episodic Memory)**: Remembering user preferences across sessions is episodic memory. The preference was expressed in a past conversation and needs to persist.
2. **A (Short-term Memory)**: Tracking what was said earlier in the current conversation is handled by the context window.
3. **C (Long-term Memory)**: Searching through thousands of documents for relevant information is exactly what vector store retrieval does.

## Summary and Wrap-Up

[TIME: 1:50 to 2:00]

# Summary: Three Pillars of Advanced Agent Design

Let us recap what we covered today.

### 1. ReAct (Reason + Act)
- Agents interleave **thinking** with **tool use** in a loop: Thought, Action, Observation
- Superior to pure Chain-of-Thought for tasks requiring external information
- Adaptive: each step is informed by the result of the previous step
- We built an e-commerce support agent with three custom tools and saw it handle single-hop, multi-hop, and edge-case queries
- Best suited for research tasks, multi-hop question answering, and dynamic problem solving

### 2. Reflection (Judge-Revise)
- Agents evaluate and improve their own outputs through a critique loop
- The Generator produces, the Judge critiques, and the Generator revises
- We built this with the raw Gemini SDK: three simple functions (generate, judge, revise) composed into a loop
- We verified the generated code actually runs correctly after reflection
- Most effective when errors are the kind that become obvious upon review
- Diminishing returns after 2 to 3 iterations in most cases

### 3. Memory Systems
- **Short-term memory**: Context window. Limited, fast, ephemeral. (Tesla sensor buffer analogy)
- **Episodic memory**: Conversation history. Medium-term, tracks interaction patterns. (Salesforce CRM analogy)
- **Long-term memory**: Vector stores. Persistent, scalable, requires embedding and retrieval. (Spotify recommendation index analogy)
- Relevance scoring via cosine similarity determines which memories get retrieved
- Production agents typically combine all three memory types

### How These Patterns Combine

In production systems, these patterns are rarely used in isolation:

- **ReAct + Memory**: An agent that reasons and acts while pulling relevant context from long-term memory. Example: a customer support agent that searches the knowledge base (ReAct) while remembering the customer's past tickets (Memory).

- **ReAct + Reflection**: An agent that takes actions in the world and then reflects on whether its approach is working. Example: an autonomous coding agent that writes code (ReAct with file tools), tests it, and then reflects on test failures to revise its approach.

- **All Three Combined**: The most capable agents (like advanced versions of Devin or AutoGPT) use all three patterns. They reason and act with tools, maintain memory across long tasks, and reflect on their progress to self-correct.

Understanding these patterns gives you the architectural vocabulary to design agents that go far beyond simple prompt-and-response interactions.



